# RQ1 — effort-reduction curve under a forest-only background (robustness check)

**Purpose.** `effort_reduction_curve.ipynb` found Random Forest and
Logistic Regression already near-ceiling at 0% expert data (Boyce
≈0.99), flat up to 90%, only dropping at 100%. Before accepting "no
partial amount of expert data earns a visible benefit" as a finding, I
checked whether this is a bug (it is not — `src/evaluation.py` is
correct, there is no train/test leakage, see chat) and whether it is a
background-composition artifact: the shared background pool (2 000
uniform random points) is ~29% non-forest land cover (roads,
agriculture, water), while the acacia presences are ~92% forest, so
part of the apparent separability may just be "forest vs. not forest,"
not "acacia vs. other forest" — an easier problem that a harder,
habitat-matched background would not inflate.

This notebook re-runs the full curve against a **forest-only background
pool**: same construction as
[`one_class_method_selection.ipynb`](one_class_method_selection.ipynb)'s
shared pool (uniform random points in the pilot bbox, 50 m exclusion
buffer around every known acacia presence), with one added filter —
a point only counts if it falls inside a COS 2023 **forest-type**
polygon: eucalyptus, pine, "other forest," or shrubland. Points in
artificial, agriculture, water/bare **and acacia-forest (COS)** land
cover are excluded — the last one because keeping a COS-labelled
"acacia forest" point as a "not acacia" background point would be a
labelling contradiction, not a harder test.

**Status: exploratory / not yet a design decision.** Nothing here
replaces the shared background pool used elsewhere in RQ1 — this is a
one-off robustness check on the effort-reduction-curve finding, run at
my own initiative in response to a plausibility question, not yet
confirmed with the supervisor either way.


> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> In this notebook "RQ1" is the effort-reduction curve, which is still
> RQ1.

## 1. Load the forest-only background pool

Same rejection-sampling pipeline as the shared background pool, plus a
COS 2023 land-cover filter. COS is read bbox-chunked per 10 km block
(same pattern as `expert_polygons.ipynb`'s `cos_group` classification),
applied to each batch of candidates that already survived the
exclusion buffer. Target: N = 2 000, same as the existing pool, so this
is a clean swap-in comparison rather than confounding pool size with
background difficulty. Saved separately
(`background_points_forest_only.npz`) — the original pool is untouched
and still used everywhere else in RQ1.

The pool is built in
[`background_pools.ipynb`](../05_data_integration/background_pools.ipynb)
(`src.background.sample_forest_only`); this notebook only reads it,
after checking it against `data/processed_manifest.csv`.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.background import load_background
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)

# Built in notebooks/05_data_integration/background_pools.ipynb; load_background() checks the
# file and its inputs against data/processed_manifest.csv before loading.
bgf = load_background(ROOT, "forest_only")
bg_x, bg_y, bg_emb = bgf["x"], bgf["y"], bgf["emb"]
print(f"{len(bg_emb):,} forest-only background points with valid embeddings")
print(pd.Series(bgf["cos_group"]).value_counts())

## 2. Load embeddings and presences

Identical to `effort_reduction_curve.ipynb` Section 1, except the
background pool loaded above is the forest-only one. Citizen-science
and expert presence extraction, and the polygon-level expert index, are
unaffected by the background swap.


In [ ]:
dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
cs_x, cs_y, cs_emb, cs_n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)

expert_gdf = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)
pilot_points = gpd.GeoDataFrame(pilot_df, geometry=gpd.points_from_xy(pilot_df["x"], pilot_df["y"]), crs=32629)
joined = gpd.sjoin(pilot_points, expert_gdf[["geometry"]], predicate="within", how="inner")
joined = joined[~joined.index.duplicated(keep="first")]
expert_idx = joined.index.values
expert_poly_id_all = joined["index_right"].values
expert_emb_all = pilot_emb[expert_idx]
valid = ~np.isnan(expert_emb_all).any(axis=1)
expert_x, expert_y = pilot_df["x"].values[expert_idx][valid], pilot_df["y"].values[expert_idx][valid]
expert_emb = expert_emb_all[valid]
expert_poly_id = expert_poly_id_all[valid]

cs_block = block_id(cs_x, cs_y)
expert_block = block_id(expert_x, expert_y)
bg_block = block_id(bg_x, bg_y)

print(f"citizen-science presences: {len(cs_emb):,} (matched {cs_n_matched}/{len(dealbata)})")
print(f"expert presence pixels: {len(expert_emb):,}, across {len(np.unique(expert_poly_id)):,} polygons")
print(f"forest-only background points: {len(bg_emb):,}")


## 3. Effort order, mixing levels, fixed test blocks

Same seed-0 polygon order and same per-seed test-block draw as
`effort_reduction_curve.ipynb` (test blocks come from citizen-science ∪
expert presence blocks, not from the background — unaffected by the
background swap), so the only thing that differs from the original
curve run is which background points are used for training and
evaluation.


In [ ]:
LEVELS = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

poly_ids_available = np.unique(expert_poly_id)
poly_order = np.random.default_rng(0).permutation(poly_ids_available)

def included_polygons(level_pct):
    if level_pct >= 100:
        return poly_ids_available
    n_incl = round(level_pct / 100 * len(poly_order))
    return poly_order[:n_incl]

candidate_blocks = sorted(set(cs_block) | set(expert_block))
SEEDS = range(1, 6)
test_blocks_by_seed = {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    test_blocks_by_seed[seed] = set(
        rng.choice(candidate_blocks, size=max(1, round(0.2 * len(candidate_blocks))), replace=False)
    )

print(f"{len(candidate_blocks)} candidate blocks (citizen-science ∪ expert presences)")
for seed, tb in test_blocks_by_seed.items():
    bg_in_test = np.isin(bg_block, list(tb)).sum()
    print(f"seed {seed}: {len(tb)} test blocks, {bg_in_test} forest-only background points inside them")


## 4. Run One-Class SVM, Random Forest and Logistic Regression across every level and seed

Identical configuration and evaluation loop to
`effort_reduction_curve.ipynb` Section 4 — the only change is `bg_emb`
/ `bg_block` now come from the forest-only pool.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from src.evaluation import continuous_boyce

pb_configs = [
    ("RandomForest", lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
    ("LogisticRegression", lambda: LogisticRegression(class_weight="balanced", max_iter=1000)),
]

records = []
for seed in SEEDS:
    test_blocks = test_blocks_by_seed[seed]
    cs_test_mask = np.isin(cs_block, list(test_blocks))
    expert_test_mask = np.isin(expert_block, list(test_blocks))
    bg_test_mask = np.isin(bg_block, list(test_blocks))

    cs_train_emb, cs_test_emb = cs_emb[~cs_test_mask], cs_emb[cs_test_mask]
    expert_train_emb, expert_test_emb = expert_emb[~expert_test_mask], expert_emb[expert_test_mask]
    expert_train_poly = expert_poly_id[~expert_test_mask]
    bg_train_emb, bg_test_emb = bg_emb[~bg_test_mask], bg_emb[bg_test_mask]

    test_pres_emb = np.concatenate([cs_test_emb, expert_test_emb])
    if len(test_pres_emb) < 5 or len(bg_test_emb) < 5:
        continue

    for lvl in LEVELS:
        if lvl >= 100:
            train_pres_emb = expert_train_emb
        else:
            incl = included_polygons(lvl)
            train_pres_emb = np.concatenate([cs_train_emb, expert_train_emb[np.isin(expert_train_poly, incl)]])
        if len(train_pres_emb) < 5:
            continue

        scaler_oc = StandardScaler().fit(train_pres_emb)
        oc_model = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit(scaler_oc.transform(train_pres_emb))
        score_pres = oc_model.decision_function(scaler_oc.transform(test_pres_emb))
        score_bg = oc_model.decision_function(scaler_oc.transform(bg_test_emb))
        y_true = np.concatenate([np.ones(len(score_pres)), np.zeros(len(score_bg))])
        y_score = np.concatenate([score_pres, score_bg])
        records.append({
            "seed": seed, "level_pct": lvl, "classifier": "OneClassSVM",
            "n_train_presence": len(train_pres_emb),
            "roc_auc": roc_auc_score(y_true, y_score),
            "boyce": continuous_boyce(score_pres, score_bg),
        })

        X_train = np.concatenate([train_pres_emb, bg_train_emb])
        y_train = np.concatenate([np.ones(len(train_pres_emb)), np.zeros(len(bg_train_emb))])
        X_test = np.concatenate([test_pres_emb, bg_test_emb])
        y_test = np.concatenate([np.ones(len(test_pres_emb)), np.zeros(len(bg_test_emb))])
        scaler_pb = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler_pb.transform(X_train), scaler_pb.transform(X_test)

        for name, ctor in pb_configs:
            model = ctor().fit(X_train_s, y_train)
            score = model.predict_proba(X_test_s)[:, 1]
            sp, sb = score[y_test == 1], score[y_test == 0]
            records.append({
                "seed": seed, "level_pct": lvl, "classifier": name,
                "n_train_presence": len(train_pres_emb),
                "roc_auc": roc_auc_score(y_test, score),
                "boyce": continuous_boyce(sp, sb),
            })

results_df = pd.DataFrame(records)
print(f"{len(results_df)} (seed, level, classifier) rows; {results_df['seed'].nunique()} valid seeds")
results_df.head()


## 5. Aggregate and compare against the original (full) background curve


In [ ]:
summary_forest = (
    results_df.groupby(["classifier", "level_pct"])
    .agg(roc_auc_mean=("roc_auc", "mean"), roc_auc_sd=("roc_auc", "std"),
         boyce_mean=("boyce", "mean"), boyce_sd=("boyce", "std"),
         n_seeds=("seed", "nunique"))
    .reset_index()
)
summary_forest["background"] = "forest_only"
summary_forest.round(3)


Original (full, 2 000-point uniform random) background results, copied
from `effort_reduction_curve.ipynb`'s Section 5 output, for a direct
side-by-side comparison — not recomputed here, so this cell has no
dependency on that notebook's objects, only its already-reported
numbers.


In [ ]:
_original_rows = [
("LogisticRegression",0,0.873,0.032,0.971,0.014),("LogisticRegression",10,0.886,0.032,0.978,0.016),
("LogisticRegression",20,0.891,0.032,0.964,0.028),("LogisticRegression",30,0.893,0.032,0.969,0.028),
("LogisticRegression",40,0.894,0.032,0.949,0.020),("LogisticRegression",50,0.895,0.032,0.966,0.029),
("LogisticRegression",60,0.895,0.032,0.940,0.033),("LogisticRegression",70,0.895,0.032,0.947,0.035),
("LogisticRegression",80,0.896,0.032,0.955,0.026),("LogisticRegression",90,0.895,0.032,0.955,0.041),
("LogisticRegression",100,0.822,0.043,0.897,0.076),
("OneClassSVM",0,0.574,0.029,0.628,0.213),("OneClassSVM",10,0.607,0.033,0.745,0.196),
("OneClassSVM",20,0.627,0.036,0.767,0.266),("OneClassSVM",30,0.634,0.033,0.724,0.261),
("OneClassSVM",40,0.639,0.033,0.689,0.256),("OneClassSVM",50,0.648,0.036,0.675,0.285),
("OneClassSVM",60,0.655,0.035,0.657,0.279),("OneClassSVM",70,0.655,0.036,0.671,0.290),
("OneClassSVM",80,0.664,0.038,0.681,0.295),("OneClassSVM",90,0.665,0.038,0.671,0.290),
("OneClassSVM",100,0.765,0.061,0.667,0.237),
("RandomForest",0,0.879,0.027,0.993,0.011),("RandomForest",10,0.880,0.028,0.988,0.009),
("RandomForest",20,0.882,0.027,0.994,0.008),("RandomForest",30,0.880,0.030,0.984,0.018),
("RandomForest",40,0.880,0.028,0.994,0.008),("RandomForest",50,0.880,0.027,0.990,0.010),
("RandomForest",60,0.879,0.029,0.979,0.019),("RandomForest",70,0.879,0.028,0.981,0.022),
("RandomForest",80,0.878,0.028,0.985,0.020),("RandomForest",90,0.878,0.028,0.983,0.014),
("RandomForest",100,0.801,0.041,0.917,0.036),
]
summary_full = pd.DataFrame(_original_rows, columns=["classifier", "level_pct", "roc_auc_mean", "roc_auc_sd", "boyce_mean", "boyce_sd"])
summary_full["background"] = "full"

compare = pd.concat([summary_full, summary_forest[summary_full.columns]], ignore_index=True)
pivot = compare.pivot_table(index=["classifier", "level_pct"], columns="background", values=["roc_auc_mean", "boyce_mean"])
pivot["roc_auc_mean", "drop"] = pivot["roc_auc_mean", "full"] - pivot["roc_auc_mean", "forest_only"]
pivot["boyce_mean", "drop"] = pivot["boyce_mean", "full"] - pivot["boyce_mean", "forest_only"]
pivot.round(3)


## 6. Plot: full vs. forest-only background, by classifier

Same two-panel layout (ROC-AUC, Boyce) and fixed classifier colours as
`effort_reduction_curve.ipynb`; the forest-only run is drawn with a
dashed line of the same colour, so each classifier's own full-vs-forest
gap is easy to read without crowding the legend.


In [ ]:
import matplotlib.pyplot as plt

COLORS = {"OneClassSVM": "#2a78d6", "RandomForest": "#eb6834", "LogisticRegression": "#1baf7a"}
MARKERS = {"OneClassSVM": "o", "RandomForest": "s", "LogisticRegression": "^"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)

for metric, ax, title in [("roc_auc_mean", axes[0], "ROC-AUC"), ("boyce_mean", axes[1], "Continuous Boyce index")]:
    for clf, color in COLORS.items():
        for bgname, ls, alpha in [("full", "-", 1.0), ("forest_only", "--", 0.85)]:
            sub = compare[(compare["classifier"] == clf) & (compare["background"] == bgname)].sort_values("level_pct")
            ax.plot(sub["level_pct"], sub[metric], color=color, marker=MARKERS[clf], linestyle=ls,
                    markersize=5, linewidth=2, alpha=alpha,
                    label=f"{clf} ({bgname})" if metric == "roc_auc_mean" else None)
    ax.set_title(title)
    ax.set_xlabel("% of expert polygons added to citizen-science base")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=0)

axes[0].set_ylabel("score (mean over 5 seeds)")
axes[0].legend(frameon=False, loc="lower right", fontsize=8)
fig.suptitle("RQ1 effort-reduction curve — full vs. forest-only background", y=1.02)
fig.tight_layout()
plt.savefig("/tmp/forest_bg_curve_plot.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Reading the comparison (2026-11-07)

**The "flat curve, no visible benefit from partial expert data" finding
survives a much harder background — and for Random Forest it holds up
slightly better, not worse.**

- **Random Forest.** ROC-AUC actually *rose* under the forest-only
  background at every mixing level (e.g. 0.879 → 0.934 at 0%, 0.878 →
  0.933 at 90%), and the curve stayed just as flat across 0–90% as
  under the full background. Continuous Boyce stayed ~0.98–0.99 under
  both. At 100% (expert-only training, no citizen science), ROC-AUC
  also rose (0.801 → 0.885) and Boyce dropped only marginally (0.917 →
  0.905). **Restricting the negative class to ecologically matched
  forest types (eucalyptus, pine, other forest, shrubland — excluding
  the COS-labelled acacia class as a labelling contradiction) does not
  expose any hidden weakness.** The near-ceiling score and the flat
  shape are not an artifact of an easy, non-forest background.
- **Logistic Regression.** Same overall pattern (ROC-AUC up ~0.04–0.06
  across 0–90%, flat shape preserved), with one exception worth
  flagging: at the 100% (expert-only) point its Boyce index collapses
  from 0.897 (full background) to 0.567 (forest-only), even though
  ROC-AUC there still improves (0.822 → 0.870). Ranking quality (AUC)
  stays fine; the Boyce index — which is threshold/density-sensitive —
  does not, specifically at the smallest, most extreme training-set
  point. Flagged as a caveat, not folded into the headline conclusion.
- **One-Class SVM behaves very differently.** ROC-AUC collapses to
  near chance at low mixing levels (0.515 at 0%, vs. 0.574 under the
  full background) and Boyce goes **negative** at 0% (-0.039, vs. 0.628
  under the full background) — the presence-only decision boundary is
  barely better than noise at telling acacia apart from other forest
  types once the "easy" non-forest background is removed. This
  strongly suggests One-Class SVM's earlier, more moderate scores
  (ROC-AUC 0.57–0.77 across the curve) were substantially inflated by
  the background's non-forest component — it was partly succeeding by
  recognising "this clearly isn't vegetation" rather than learning
  anything acacia-specific, exactly the risk informally expected of a
  presence-only method without negative supervision. The monotonic
  *improvement* with more expert data is still present (ROC-AUC
  0.515→0.754 from 0% to 100%), actually larger in relative terms, but
  off a much lower floor.

**Answer to "couldn't it be a mistake?" (the question that motivated
this whole check):** no — across the code review of
`continuous_boyce()`, the leakage distance check, the land-cover
composition comparison, and now this forest-only robustness re-run of
the *entire* curve, the Random Forest / Logistic Regression
near-ceiling result and the "no partial benefit from expert data"
reading hold up. The one real update: One-Class SVM's apparent partial
competitiveness earlier in the curve now looks like it was
substantially a background-composition artifact specific to that one
model — not something that touches the presence-vs-background
classifiers (Random Forest, proposed default for RQ1).

**Caveats, carried over from the main curve notebook plus new ones
specific to this check:**
- Forest-only background is still N=2,000 (matched to the original
  pool's size, not independently tuned or enlarged).
- The forest-type COS grouping (eucalyptus / pine / other forest /
  shrubland) is my own grouping of COS 2023 codes, not yet confirmed
  by the supervisor.
- The Logistic Regression Boyce collapse at 100% is noted but not
  further investigated here.
- This is a one-off robustness check layered on top of an
  already-unconfirmed shared background-pool design
  (`data/README.md`) — it does not replace that pool, only stress-tests
  the curve's conclusion against a harder alternative.
- No hyperparameter re-tuning for either background; same
  configurations as every other RQ1 notebook so far.
